# Pix2pixRAD input preprocessing
- images are 3-channel, with adjustable input conditions
- inputs (left) are concatenated with the target (right)
- data preprocessed according to the needs of Conte et al. (2021)
- https://github.com/giemmecci/pix2pixRAD

In [ ]:
import os
import shutil
import numpy as np
from PIL import Image

# ── Config ────────────────────────────────────────────────────────────────────
split_source_root = "/path/to/your/split_root"         
pix2pix_rad_root = "/path/to/pix2pixRAD/original/output"                 
paired_root = "/path/to/pix2pixRAD/paired/output"       

splits = ["train", "validation", "test"]
subfolders = ["t1n", "t1c", "seg_mod", "t2f"]




In [ ]:
# Copy subfolders into pix2pixRAD

for split in splits:
    src_root = os.path.join(split_source_root, split)
    dst_root = os.path.join(pix2pix_rad_root, split)

    for patient in sorted(os.listdir(src_root)):
        patient_src = os.path.join(src_root, patient)
        if not os.path.isdir(patient_src):
            continue
        patient_dst = os.path.join(dst_root, patient)
        os.makedirs(patient_dst, exist_ok=True)

        for sub in subfolders:
            src_sub = os.path.join(patient_src, sub)
            dst_sub = os.path.join(patient_dst, sub)
            if os.path.exists(src_sub):
                shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
            else:
                print(f" Missing {sub} for {split}/{patient}")
        print(f" {split}/{patient}")


## create input configs
- default t1n, seg, t1n 
- uncomment t2f to create the t1n, seg, flair input scheme
  

In [ ]:

# Choose  multichannel combination by commenting/uncommenting:
# if only t1n, then adjust code to simplify for a single input across 3 channels (RGB)

MULTICHANNEL_MODE = "t1n_seg_t1n"   # R=t1n, G=seg_mod, B=t1n
# MULTICHANNEL_MODE = "t1n_seg_t2f" # R=t1n, G=seg_mod, B=t2f


# Multichannel inputs
def create_multichannel(src_root, out_root):
    os.makedirs(out_root, exist_ok=True)
    for patient in sorted(os.listdir(src_root)):
        patient_path = os.path.join(src_root, patient)
        if not os.path.isdir(patient_path):
            continue

        t1n_path = os.path.join(patient_path, "t1n")
        seg_path = os.path.join(patient_path, "seg_mod")
       # t2f_path = os.path.join(patient_path, "t2f")       # only used in t1n_seg_t2f mode

        t1n_files = sorted(f for f in os.listdir(t1n_path) if f.endswith(".png"))
        seg_files = sorted(f for f in os.listdir(seg_path) if f.endswith(".png"))
       # t2f_files = sorted(f for f in os.listdir(t2f_path) if f.endswith(".png"))

        if len(t1n_files) != len(seg_files):
            print(f" {patient} — t1n/seg_mod count mismatch, using minimum")

        patient_out = os.path.join(out_root, patient, MULTICHANNEL_MODE)
        os.makedirs(patient_out, exist_ok=True)

        file_iter = zip(t1n_files, seg_files) #, t2f_files) 

        for t1n_f, seg_f, t2f_f in file_iter:   # include t2f if needed
            t1n_img = np.array(Image.open(os.path.join(t1n_path, t1n_f))).astype(np.uint8)
            seg_img = np.array(Image.open(os.path.join(seg_path, seg_f))).astype(np.uint8)
        #   t2f_img = np.array(Image.open(os.path.join(t2f_path, t2f_f))).astype(np.uint8)

            rgb = np.stack([t1n_img, seg_img, t1n_img], axis=-1) # add t2f if needed

            Image.fromarray(rgb).save(os.path.join(patient_out, t1n_f))

        print(f"  [OK] {patient} ({len(t1n_files)} slices)")

for split in splits:
    print(f"\n  -- {split} --")
    create_multichannel(
        src_root = os.path.join(pix2pix_rad_root, split),
        out_root = os.path.join(paired_root, split)
    )


### Create paired images
- required for the pix2pix framework (left = conditioned input, right = target)

In [ ]:
# multichannel on the left, target on the right
def create_paired(split_pix2pix_root, split_paired_root):
    for patient in sorted(os.listdir(split_paired_root)):
        patient_path = os.path.join(split_paired_root, patient)
        if not os.path.isdir(patient_path):
            continue

        multi_path = os.path.join(patient_path, MULTICHANNEL_MODE)  
        t1c_path   = os.path.join(split_pix2pix_root, patient, "t1c")
        paired_out = os.path.join(patient_path, "paired")

        if not (os.path.exists(multi_path) and os.path.exists(t1c_path)):
            print(f" {patient} — missing multichannel or t1c")
            continue

        os.makedirs(paired_out, exist_ok=True)
        multi_files = sorted(f for f in os.listdir(multi_path) if f.endswith(".png"))
        t1c_files   = sorted(f for f in os.listdir(t1c_path)   if f.endswith(".png"))
        if len(multi_files) != len(t1c_files):
            print(f" {patient} — multichannel/t1c count mismatch, using minimum")

        for m_f, t_f in zip(multi_files, t1c_files):
            multi_img = Image.open(os.path.join(multi_path, m_f))
            t1c_img   = Image.open(os.path.join(t1c_path, t_f)).convert("RGB")
            w, h = multi_img.size
            paired = Image.new("RGB", (w * 2, h))
            paired.paste(multi_img, (0, 0))
            paired.paste(t1c_img,   (w, 0))
            paired.save(os.path.join(paired_out, m_f))

        print(f" {patient} ({len(multi_files)} pairs)")


# Now flatten these folders
def flatten(split_paired_root):
    for patient in sorted(os.listdir(split_paired_root)):
        paired_path = os.path.join(split_paired_root, patient, "paired")
        if not os.path.isdir(paired_path):
            continue
        png_files = [f for f in os.listdir(paired_path) if f.endswith(".png")]
        for fname in png_files:
            src = os.path.join(paired_path, fname)
            dst = os.path.join(split_paired_root, f"{patient}_{fname}")
            shutil.move(src, dst)
        os.rmdir(paired_path)
        print(f" {patient} ({len(png_files)} files)")

for split in splits:
    flatten(os.path.join(paired_root, split))
